# Reproducing Figure 3 (panels F1, C1, C2) of Keskar et al., ICLR 2017
*On Large-Batch Training for Deep Learning: Generalization Gap and Sharp Minima*

For each network we

1. build the model once with a fixed seed and save that initialization **x₀**;
2. train a **small-batch (SB)** copy (batch 256) and a **large-batch (LB)** copy (batch = 10% of the training set) from the *same* x₀, both with ADAM, until the training loss stops improving;
3. evaluate **f(α·x*_LB + (1−α)·x*_SB)** for α ∈ [−1, 2] on the full training and test sets, interpolating **all** weights *and* the batch-norm running statistics with the same α;
4. plot cross-entropy (left axis, blue) and accuracy (right axis, red); solid = train, dashed = test. α = 0 is the SB minimizer, α = 1 the LB minimizer.

**Kaggle settings:** turn on *Internet* (MNIST / CIFAR-10 are downloaded with torchvision) and select a **GPU** accelerator (P100 recommended: the LB VGG run uses batches of 5,000 images). Trained models and interpolation results are cached in `/kaggle/working`, so re-running the notebook skips finished work.

Paper details that are reproduced as stated: architectures of Appendix B, data used without preprocessing, SB batch 256, LB batch 10% of the data, ADAM for both regimes, training run until the loss ceases to improve, interpolation range α ∈ [−1, 2], one SB/LB pair drawn at random from the trials. Details the paper does not state (learning rate, pooling stride/padding, the exact stopping rule, number of α points) are set in the configuration cell and flagged there.

In [ ]:
import os, json, time, math, random, copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
import torchvision
import matplotlib.pyplot as plt

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ON_KAGGLE = os.path.isdir("/kaggle/working")
OUT_DIR = "/kaggle/working/keskar_fig3" if ON_KAGGLE else "./keskar_fig3"
DATA_DIR = "/kaggle/working/data" if ON_KAGGLE else "./data"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

torch.backends.cudnn.benchmark = True
print("torch", torch.__version__, "| device:", DEVICE,
      torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")

## Configuration

In [ ]:
CFG = dict(
    # ---- stated in the paper ----
    sb_batch=256,            # "256 data points for small-batch experiments"
    lb_fraction=0.10,        # "10% of the training data as batch size for the large-batch experiments"
    alpha_min=-1.0, alpha_max=2.0,
    n_trials=1,              # paper ran 5 trials and plotted one random SB/LB pair; raise to 5 if you have time

    # ---- NOT stated in the paper (our choices) ----
    lr=1e-3,                 # ADAM default step size, used for both SB and LB
    n_alphas=25,             # alpha grid spacing 0.125 (alpha = -0.25 and 1.25 are on the grid)
    # "trained ... until the loss function ceased to improve": we stop when the epoch-mean
    # training loss has not dropped by more than rel_tol (relative) for `patience` epochs,
    # or when max_epochs is reached.
    rel_tol=0.01,
    sb_patience=10, lb_patience=25,  # LB takes ~40x fewer steps per epoch, so it gets more epochs of patience
    eval_every=5,            # epochs between full train/test evaluations (training curves)
    eval_batch=1000,
    base_seed=0,
)

EXPERIMENTS = {
    "F1": dict(dataset="mnist",   sb_max_epochs=100, lb_max_epochs=1000),
    "C1": dict(dataset="cifar10", sb_max_epochs=100, lb_max_epochs=500),
    "C2": dict(dataset="cifar10", sb_max_epochs=100, lb_max_epochs=500),
}

QUICK_TEST = False   # True: tiny epoch budgets just to check the pipeline end to end
if QUICK_TEST:
    for e in EXPERIMENTS.values():
        e["sb_max_epochs"], e["lb_max_epochs"] = 2, 4
    CFG["n_alphas"] = 7

## Data
The paper uses MNIST and CIFAR-10 "without any pre-processing"; we only convert pixels to floats in [0, 1]. The whole data set is kept on the GPU and batches are drawn by random permutation each epoch (fixed batch size, randomly sampled).

In [ ]:
_DATA_CACHE = {}

def load_dataset(name):
    if name in _DATA_CACHE:
        return _DATA_CACHE[name]
    if name == "mnist":
        tr = torchvision.datasets.MNIST(DATA_DIR, train=True, download=True)
        te = torchvision.datasets.MNIST(DATA_DIR, train=False, download=True)
        def prep(ds):
            x = ds.data.float().div(255.).reshape(len(ds.data), -1)       # 784-dim input for F1
            y = ds.targets.long()
            return x, y
    elif name == "cifar10":
        tr = torchvision.datasets.CIFAR10(DATA_DIR, train=True, download=True)
        te = torchvision.datasets.CIFAR10(DATA_DIR, train=False, download=True)
        def prep(ds):
            x = torch.from_numpy(ds.data).float().div(255.).permute(0, 3, 1, 2).contiguous()  # N,3,32,32
            y = torch.tensor(ds.targets, dtype=torch.long)
            return x, y
    else:
        raise ValueError(name)
    Xtr, Ytr = prep(tr)
    Xte, Yte = prep(te)
    d = dict(Xtr=Xtr.to(DEVICE), Ytr=Ytr.to(DEVICE), Xte=Xte.to(DEVICE), Yte=Yte.to(DEVICE))
    _DATA_CACHE[name] = d
    print(f"{name}: train {tuple(Xtr.shape)}, test {tuple(Xte.shape)}")
    return d

## Networks (Appendix B)

* **F1**: 784 → 5 × [512, batch-norm, ReLU] → 10 (softmax via cross-entropy).
* **C1** (AlexNet-like): 2 × {conv 64 filters 5×5 stride 2 → BN → ReLU → MaxPool(3)} → dense 384 → dense 192 → 10. BN on all layers, ReLU, dropout with retention 0.5 on the two dense layers. *Unstated:* conv padding 2, max-pool stride 2 / padding 1 (the usual AlexNet/CIFAR pooling).
* **C2** (VGG-like): conv 3×3 stride 1 stacks 2×64, 2×128, 3×256, 3×512, 3×512, MaxPool(2) after each stack → dense 512 → 10, with BN, ReLU, and dropout 0.5 on the dense layer "as in B.3". *Unstated:* conv padding 1.

PyTorch's default (uniform) initialization is used. Each conv/linear+BN+ReLU unit is a separate module so that, for the 5,000-image LB batches, activations can be recomputed in the backward pass (gradient checkpointing). This saves memory **without** changing the computation: batch-norm still normalizes over the full 5,000-image batch (no "ghost batches"), and the second running-statistics update that recomputation would cause is undone (see `train_one`).

In [ ]:
def fc_unit(i, o):
    return nn.Sequential(nn.Linear(i, o), nn.BatchNorm1d(o), nn.ReLU(inplace=True))

def conv_unit(i, o, k, s, p):
    return nn.Sequential(nn.Conv2d(i, o, k, stride=s, padding=p), nn.BatchNorm2d(o), nn.ReLU(inplace=True))

class Net(nn.Module):
    # units: run in sequence (optionally checkpointed); head: dense part, never checkpointed
    def __init__(self, units, head):
        super().__init__()
        self.units = nn.ModuleList(units)
        self.head = head
        self.use_ckpt = False
    def forward(self, x):
        for u in self.units:
            if self.use_ckpt and self.training and torch.is_grad_enabled():
                x = checkpoint(u, x, use_reentrant=False)
            else:
                x = u(x)
        return self.head(x)

def make_F1():
    units = [fc_unit(784, 512)] + [fc_unit(512, 512) for _ in range(4)]
    return Net(units, nn.Linear(512, 10))

def make_C1(n_classes=10):
    units = [
        nn.Sequential(conv_unit(3, 64, 5, 2, 2),  nn.MaxPool2d(3, stride=2, padding=1)),   # 32 -> 16 -> 8
        nn.Sequential(conv_unit(64, 64, 5, 2, 2), nn.MaxPool2d(3, stride=2, padding=1)),   # 8 -> 4 -> 2
    ]
    head = nn.Sequential(
        nn.Flatten(),
        nn.Linear(64 * 2 * 2, 384), nn.BatchNorm1d(384), nn.ReLU(inplace=True), nn.Dropout(0.5),
        nn.Linear(384, 192),        nn.BatchNorm1d(192), nn.ReLU(inplace=True), nn.Dropout(0.5),
        nn.Linear(192, n_classes),
    )
    return Net(units, head)

def make_C2(n_classes=10):
    units, c_in = [], 3
    for n_conv, c_out in [(2, 64), (2, 128), (3, 256), (3, 512), (3, 512)]:
        for j in range(n_conv):
            u = conv_unit(c_in, c_out, 3, 1, 1)
            if j == n_conv - 1:
                u = nn.Sequential(u, nn.MaxPool2d(2))
            units.append(u)
            c_in = c_out
    head = nn.Sequential(                                         # 32 -> 1 after five pools
        nn.Flatten(),
        nn.Linear(512, 512), nn.BatchNorm1d(512), nn.ReLU(inplace=True), nn.Dropout(0.5),
        nn.Linear(512, n_classes),
    )
    return Net(units, head)

MODEL_FNS = {"F1": make_F1, "C1": make_C1, "C2": make_C2}

for k, fn in MODEL_FNS.items():
    print(k, sum(p.numel() for p in fn().parameters()), "parameters")

## Training
Both regimes use ADAM with the same step size, starting from the identical saved initialization. Training stops when the epoch-mean training loss stops improving (rule in `CFG`), capped at `max_epochs`. Every `eval_every` epochs the full train/test accuracy is recorded, which also lets us check (as in the paper's Figure 2) that the gap is not over-fitting: the test curve should plateau rather than peak and decay.

In [ ]:
def bn_buffers(model):
    return {k: v.clone() for k, v in model.state_dict().items()
            if k.endswith(("running_mean", "running_var", "num_batches_tracked"))}

@torch.no_grad()
def evaluate(model, X, Y, bs):
    model.eval()
    tot_loss, correct = 0.0, 0
    for i in range(0, X.shape[0], bs):
        out = model(X[i:i + bs]).float()
        tot_loss += F.cross_entropy(out, Y[i:i + bs], reduction="sum").item()
        correct += (out.argmax(1) == Y[i:i + bs]).sum().item()
    n = X.shape[0]
    return tot_loss / n, 100.0 * correct / n

def train_one(model, data, batch_size, max_epochs, patience, seed, tag):
    model.to(DEVICE)
    Xtr, Ytr = data["Xtr"], data["Ytr"]
    n = Xtr.shape[0]
    # recompute activations in backward only when the batch is large (memory); exact either way
    model.use_ckpt = batch_size >= 2048 and DEVICE.type == "cuda"
    opt = torch.optim.Adam(model.parameters(), lr=CFG["lr"])
    gen = torch.Generator().manual_seed(seed)
    torch.manual_seed(seed)                      # dropout masks
    hist = dict(epoch=[], train_loss_running=[], eval_epoch=[], train_acc=[], test_acc=[])
    best, since_best, t0 = float("inf"), 0, time.time()
    for ep in range(1, max_epochs + 1):
        model.train()
        perm = torch.randperm(n, generator=gen).to(DEVICE)
        run_loss = torch.zeros((), device=DEVICE)
        for i in range(0, n, batch_size):
            idx = perm[i:i + batch_size]
            loss = F.cross_entropy(model(Xtr[idx]), Ytr[idx])
            opt.zero_grad(set_to_none=True)
            if model.use_ckpt:
                # forward already updated BN running stats once; the recomputation inside
                # backward would update them a second time -> restore the post-forward values
                saved = bn_buffers(model)
                loss.backward()
                model.load_state_dict(saved, strict=False)
            else:
                loss.backward()
            opt.step()
            run_loss += loss.detach() * idx.numel()
        ep_loss = run_loss.item() / n
        if not math.isfinite(ep_loss):
            raise RuntimeError(f"{tag}: training loss became {ep_loss}")
        hist["epoch"].append(ep); hist["train_loss_running"].append(ep_loss)
        if ep % CFG["eval_every"] == 0 or ep == 1:
            _, tr_acc = evaluate(model, Xtr, Ytr, CFG["eval_batch"])
            _, te_acc = evaluate(model, data["Xte"], data["Yte"], CFG["eval_batch"])
            hist["eval_epoch"].append(ep); hist["train_acc"].append(tr_acc); hist["test_acc"].append(te_acc)
            print(f"  [{tag}] ep {ep:4d}  loss {ep_loss:.5f}  train {tr_acc:6.2f}%  test {te_acc:6.2f}%  "
                  f"({time.time() - t0:.0f}s)")
        if ep_loss < best * (1 - CFG["rel_tol"]):
            best, since_best = ep_loss, 0
        else:
            since_best += 1
            if since_best >= patience:
                print(f"  [{tag}] stopped at epoch {ep}: no {CFG['rel_tol']:.0%} improvement for {patience} epochs")
                break
    model.use_ckpt = False
    hist["epochs_trained"] = ep
    return hist

## Interpolation and evaluation along the line
`interpolate_state` blends every floating-point tensor of the two state dicts — weights, biases, BN affine parameters **and** BN running mean/variance — with the same α. In the extrapolated region (α < 0 or α > 1) a blended running variance can become negative; such entries are clamped to 0 (BN still divides by √(var + ε)), and the fraction clamped is reported. Evaluation is in `eval()` mode, so BN uses these interpolated running statistics and dropout is off.

In [ ]:
def interpolate_state(sd_s, sd_l, a):
    out, n_clamped, n_var = {}, 0, 0
    for k, vs in sd_s.items():
        vl = sd_l[k]
        if vs.is_floating_point():
            v = (1.0 - a) * vs + a * vl
            if k.endswith("running_var"):
                n_var += v.numel()
                n_clamped += (v < 0).sum().item()
                v = v.clamp_min(0.0)
        else:
            v = vs.clone()               # num_batches_tracked (unused in eval mode)
        out[k] = v
    return out, (n_clamped / max(n_var, 1))

@torch.no_grad()
def line_scan(model_fn, sd_s, sd_l, data, alphas):
    model = model_fn().to(DEVICE)
    res = dict(alpha=[], train_loss=[], test_loss=[], train_acc=[], test_acc=[], var_clamped_frac=[])
    for a in alphas:
        sd, frac = interpolate_state(sd_s, sd_l, float(a))
        model.load_state_dict(sd)
        trl, tra = evaluate(model, data["Xtr"], data["Ytr"], CFG["eval_batch"])
        tel, tea = evaluate(model, data["Xte"], data["Yte"], CFG["eval_batch"])
        for key, val in zip(res, [float(a), trl, tel, tra, tea, frac]):
            res[key].append(val)
        print(f"  alpha {a:+.3f}: train CE {trl:9.4f} acc {tra:6.2f}% | test CE {tel:9.4f} acc {tea:6.2f}%"
              + (f" | clamped var {frac:.1%}" if frac > 0 else ""))
    return res

## Run the experiments
For every network and trial: save x₀, train SB and LB from x₀, record the distance each solution travelled from x₀ (the paper reports ‖x*_SB − x₀‖ / ‖x*_LB − x₀‖ in the range 3–10), pick one trial at random, and scan the line. Results are cached per network.

In [ ]:
def flat_params(sd, ref_keys):
    # init state dict is created on CPU, trained ones live on the GPU -> put everything on one device
    return torch.cat([sd[k].detach().float().flatten().to(DEVICE) for k in ref_keys])

def run_network(name):
    spec = EXPERIMENTS[name]
    res_path = os.path.join(OUT_DIR, f"{name}_results.json")
    if os.path.exists(res_path):
        print(f"{name}: loading cached results from {res_path}")
        with open(res_path) as f:
            return json.load(f)
    data = load_dataset(spec["dataset"])
    n_train = data["Xtr"].shape[0]
    lb_batch = int(round(CFG["lb_fraction"] * n_train))
    print(f"\n===== {name} | SB batch {CFG['sb_batch']} | LB batch {lb_batch} =====")
    trials = []
    for t in range(CFG["n_trials"]):
        seed = CFG["base_seed"] + 1000 * t
        ckpt = os.path.join(OUT_DIR, f"{name}_trial{t}.pt")
        if os.path.exists(ckpt):
            trials.append(torch.load(ckpt, map_location=DEVICE))
            print(f"trial {t}: loaded {ckpt}")
            continue
        torch.manual_seed(seed)
        init_sd = copy.deepcopy(MODEL_FNS[name]().state_dict())      # x0, shared by SB and LB

        sb = MODEL_FNS[name](); sb.load_state_dict(init_sd)
        print(f"trial {t}: small batch")
        h_sb = train_one(sb, data, CFG["sb_batch"], spec["sb_max_epochs"], CFG["sb_patience"], seed + 1, f"{name} SB")

        lb = MODEL_FNS[name](); lb.load_state_dict(init_sd)
        print(f"trial {t}: large batch")
        h_lb = train_one(lb, data, lb_batch, spec["lb_max_epochs"], CFG["lb_patience"], seed + 2, f"{name} LB")

        rec = dict(init=init_sd,
                   sb={k: v.detach().clone() for k, v in sb.state_dict().items()},
                   lb={k: v.detach().clone() for k, v in lb.state_dict().items()},
                   hist_sb=h_sb, hist_lb=h_lb)
        torch.save(rec, ckpt)
        trials.append(rec)

    summary = []
    pkeys = [k for k, _ in MODEL_FNS[name]().named_parameters()]
    for t, rec in enumerate(trials):
        x0 = flat_params(rec["init"], pkeys)
        d_sb = (flat_params(rec["sb"], pkeys) - x0).norm().item()
        d_lb = (flat_params(rec["lb"], pkeys) - x0).norm().item()
        m = MODEL_FNS[name]().to(DEVICE)
        row = dict(trial=t, dist_sb=d_sb, dist_lb=d_lb, dist_ratio=d_sb / d_lb,
                   epochs_sb=rec["hist_sb"]["epochs_trained"], epochs_lb=rec["hist_lb"]["epochs_trained"])
        for tag in ("sb", "lb"):
            m.load_state_dict(rec[tag])
            row[f"train_acc_{tag}"] = evaluate(m, data["Xtr"], data["Ytr"], CFG["eval_batch"])[1]
            row[f"test_acc_{tag}"] = evaluate(m, data["Xte"], data["Yte"], CFG["eval_batch"])[1]
        summary.append(row)
        print(f"trial {t}: SB train/test {row['train_acc_sb']:.2f}/{row['test_acc_sb']:.2f}%  "
              f"LB train/test {row['train_acc_lb']:.2f}/{row['test_acc_lb']:.2f}%  "
              f"||xSB-x0||/||xLB-x0|| = {row['dist_ratio']:.2f}")

    pick = random.Random(CFG["base_seed"]).randrange(len(trials))   # random SB/LB pair, as in the paper
    print(f"{name}: scanning the line for trial {pick}")
    alphas = np.linspace(CFG["alpha_min"], CFG["alpha_max"], CFG["n_alphas"])
    scan = line_scan(MODEL_FNS[name], trials[pick]["sb"], trials[pick]["lb"], data, alphas)
    out = dict(name=name, lb_batch=lb_batch, picked_trial=pick, scan=scan, summary=summary,
               hist_sb=trials[pick]["hist_sb"], hist_lb=trials[pick]["hist_lb"], cfg=CFG)
    with open(res_path, "w") as f:
        json.dump(out, f, indent=1)
    return out

RESULTS = {}
for name in ["F1", "C1", "C2"]:
    RESULTS[name] = run_network(name)
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

## Figure 3 (F1, C1, C2)
Same layout as the paper: left axis cross-entropy (blue), right axis accuracy in % (red), solid = training set, dashed = test set.

In [ ]:
def plot_panel(ax, scan, title):
    a = np.array(scan["alpha"])
    ax.plot(a, scan["train_loss"], "b-",  label="Train")
    ax.plot(a, scan["test_loss"],  "b--", label="Test")
    ax.set_xlabel("alpha"); ax.set_ylabel("Cross Entropy", color="b")
    ax.set_xlim(CFG["alpha_min"], CFG["alpha_max"]); ax.set_ylim(bottom=0)
    ax.tick_params(axis="y", colors="b"); ax.grid(True, linestyle=":")
    ax2 = ax.twinx()
    ax2.plot(a, scan["train_acc"], "r-")
    ax2.plot(a, scan["test_acc"],  "r--")
    ax2.set_ylim(0, 100); ax2.set_ylabel("Accuracy", color="r"); ax2.tick_params(axis="y", colors="r")
    ax.legend(loc="upper left")
    ax.set_title(title)

fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))
for ax, (name, lab) in zip(axes, [("F1", "(a) $F_1$"), ("C1", "(c) $C_1$"), ("C2", "(d) $C_2$")]):
    plot_panel(ax, RESULTS[name]["scan"], lab)
fig.suptitle("Parametric plots – linear: alpha = 0 is the SB minimizer, alpha = 1 the LB minimizer")
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "figure3_F1_C1_C2.png"), dpi=150)
plt.show()

## Checks against the paper's claims
The plot is only meaningful if the runs reproduce the paper's conditions, so we print:

* **Table 2 analogue**: both regimes should reach high training accuracy while LB has lower test accuracy.
* **Distance travelled**: ‖x*_SB − x₀‖ / ‖x*_LB − x₀‖ (paper: 3–10).
* **Sharpness along the line**: the rise in training loss when stepping 0.25 (in α units, i.e. the same distance in weight space) *outward* from each minimizer — α = 0 → −0.25 for SB and α = 1 → 1.25 for LB. The paper's figure shows the loss rising much more steeply around the LB end.
* **Training curves** (Figure 2 analogue): the test accuracy should plateau rather than peak and decay, i.e. the gap is not classical over-fitting.

In [ ]:
def at(scan, key, a):
    i = int(np.argmin(np.abs(np.array(scan["alpha"]) - a)))
    return scan[key][i], scan["alpha"][i]

print(f"{'net':4s} {'SB train':>9s} {'LB train':>9s} {'SB test':>8s} {'LB test':>8s} {'gap':>6s} "
      f"{'dist ratio':>10s} {'ep SB':>6s} {'ep LB':>6s} {'dL SB(out)':>11s} {'dL LB(out)':>11s}")
for name, r in RESULTS.items():
    s = r["summary"][r["picked_trial"]]
    sc = r["scan"]
    l0, _ = at(sc, "train_loss", 0.0);  lm, am = at(sc, "train_loss", -0.25)
    l1, _ = at(sc, "train_loss", 1.0);  lp, ap = at(sc, "train_loss", 1.25)
    print(f"{name:4s} {s['train_acc_sb']:8.2f}% {s['train_acc_lb']:8.2f}% {s['test_acc_sb']:7.2f}% "
          f"{s['test_acc_lb']:7.2f}% {s['test_acc_sb'] - s['test_acc_lb']:5.2f}  {s['dist_ratio']:10.2f} "
          f"{s['epochs_sb']:6d} {s['epochs_lb']:6d} {lm - l0:11.4f} {lp - l1:11.4f}")
    if len(r["summary"]) > 1:
        ratios = [x["dist_ratio"] for x in r["summary"]]
        gaps = [x["test_acc_sb"] - x["test_acc_lb"] for x in r["summary"]]
        print(f"      over {len(ratios)} trials: test gap {np.mean(gaps):.2f} ± {np.std(gaps):.2f}, "
              f"dist ratio {np.mean(ratios):.2f} ± {np.std(ratios):.2f}")

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, name in zip(axes, ["F1", "C1", "C2"]):
    r = RESULTS[name]
    for tag, c in (("sb", "b"), ("lb", "r")):
        h = r[f"hist_{tag}"]
        ax.plot(h["eval_epoch"], h["train_acc"], c + "--", label=f"{tag.upper()} - Training")
        ax.plot(h["eval_epoch"], h["test_acc"],  c + "-",  label=f"{tag.upper()} - Testing")
    ax.set_title(f"{name}: accuracy vs epoch"); ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy")
    ax.grid(True, linestyle=":"); ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "training_curves.png"), dpi=120)
plt.show()

### Notes on fidelity
* The paper trains "without any budget or limits, until the loss function ceased to improve". The stopping rule and epoch caps above are our concrete version of that; if the LB runs hit `lb_max_epochs` before their training accuracy is near 100%, raise the cap (delete the cached `*_trial*.pt` / `*_results.json` files in `OUT_DIR` first).
* The learning rate is not given in the paper; ADAM's default 1e-3 is used for both regimes so that batch size is the only difference.
* Exact numbers will differ from the paper (different framework, random seeds, unstated hyper-parameters); what should reproduce is the qualitative picture — comparable training loss at α = 0 and α = 1, a test-accuracy gap in favour of SB, and a much steeper rise of the loss around the LB minimizer than around the SB minimizer.